Baseline + SMOTE
Model: RandomForest
Podzial: 60/40
Random state = 30

Precision (fraud): 0.8710
Recall (fraud):    0.6438
F1 (fraud):        0.7403

In [1]:
import pandas as pd
from pandas.core.common import random_state
from sklearn.metrics import roc_auc_score, confusion_matrix, average_precision_score, classification_report

from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("../data/credit_card_frauds.csv")
print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())

  trans_date_trans_time                   merchant       category     amt  \
0   2019-01-01 00:00:44  Heller, Gutmann and Zieme    grocery_pos  107.23   
1   2019-01-01 00:00:51             Lind-Buckridge  entertainment  220.11   
2   2019-01-01 00:07:27                  Kiehn Inc    grocery_pos   96.29   
3   2019-01-01 00:09:03                Beier-Hyatt   shopping_pos    7.77   
4   2019-01-01 00:21:32                 Bruen-Yost       misc_pos    6.85   

                       city state      lat      long  city_pop  \
0                    Orient    WA  48.8878 -118.2105       149   
1                Malad City    ID  42.1808 -112.2620      4154   
2                   Grenada    CA  41.6125 -122.5258       589   
3  High Rolls Mountain Park    NM  32.9396 -105.8189       899   
4                   Freedom    WY  43.0172 -111.0292       471   

                                 job         dob  \
0  Special educational needs teacher  1978-06-21   
1        Nature conservation officer

In [3]:
print(df.describe(include="all").T)
print(df["is_fraud"].value_counts())
print(df["is_fraud"].value_counts(normalize=True))

for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

                          count  unique                               top  \
trans_date_trans_time    339607  338504               2019-12-06 20:30:44   
merchant                 339607     693                       Kilback LLC   
category                 339607      14                     gas_transport   
amt                    339607.0     NaN                               NaN   
city                     339607     176                           Phoenix   
state                    339607      13                                CA   
lat                    339607.0     NaN                               NaN   
long                   339607.0     NaN                               NaN   
city_pop               339607.0     NaN                               NaN   
job                      339607     163                Surveyor, minerals   
dob                      339607     187                        1987-04-23   
trans_num                339607  339607  14392d723bb7737606b2700ac791b7aa   

In [4]:
#Pozbywamy sie numeru transakcji
X = df.drop(["is_fraud", "trans_num"], axis=1)
y = df["is_fraud"]

In [5]:
#Stosujemy stratify by zachowac balans klas w zbiorach.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.40,
    random_state=30
)

In [6]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\ny_train:")
print(y_train.value_counts())

print("\ny_test:")
print(y_test.value_counts())

X_train: (203764, 13)
X_test: (135843, 13)

y_train:
is_fraud
0    202695
1      1069
Name: count, dtype: int64

y_test:
is_fraud
0    135130
1       713
Name: count, dtype: int64


In [7]:
#rozbicie daty na godziny, dni, miesiace i dzien tygodnia oraz liczymy wiek.
X_train = X_train.copy()
X_test = X_test.copy()

X_train["trans_date_trans_time"] = pd.to_datetime(X_train["trans_date_trans_time"])

X_test["trans_date_trans_time"] = pd.to_datetime(X_test["trans_date_trans_time"])

X_train["dob"] = pd.to_datetime(X_train["dob"])
X_test["dob"] = pd.to_datetime(X_test["dob"])

for data in [X_train, X_test]:
    data["trans_hour"] = data["trans_date_trans_time"].dt.hour
    data["trans_day"] = data["trans_date_trans_time"].dt.day
    data["trans_month"] = data["trans_date_trans_time"].dt.month
    data["trans_dayofweek"] = data["trans_date_trans_time"].dt.dayofweek

X_train["age"] = (X_train["trans_date_trans_time"].dt.year - X_train["dob"].dt.year)

X_test["age"] = (X_test["trans_date_trans_time"].dt.year - X_test["dob"].dt.year)

X_train = X_train.drop(["trans_date_trans_time", "dob"], axis=1)

X_test = X_test.drop(["trans_date_trans_time", "dob"], axis=1)

print(X_train.dtypes)

merchant            object
category            object
amt                float64
city                object
state               object
lat                float64
long               float64
city_pop             int64
job                 object
merch_lat          float64
merch_long         float64
trans_hour           int32
trans_day            int32
trans_month          int32
trans_dayofweek      int32
age                  int32
dtype: object


In [8]:
for col in X_train.select_dtypes(include="object").columns:
    print(f"{col}: {X_train[col].nunique()}")

merchant: 693
category: 14
city: 176
state: 13
job: 163


In [9]:
from imblearn.over_sampling import SMOTENC

categorical_columns = [
    "merchant",
    "category",
    "city",
    "state",
    "job"
]

smotenc = SMOTENC(
    categorical_features=categorical_columns,
    sampling_strategy="auto",
    random_state=30
)

X_train_smote, y_train_smote = smotenc.fit_resample(
    X_train,
    y_train
)

In [10]:
#Zamiana wartosci tekstowych na liczbowe
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)

X_train_encoded = encoder.fit_transform(X_train_smote[categorical_columns])

X_test_encoded = encoder.transform(X_test[categorical_columns])

In [11]:
from scipy.sparse import hstack

numeric_columns = [
    "amt",
    "lat",
    "long",
    "city_pop",
    "merch_lat",
    "merch_long",
    "trans_hour",
    "trans_day",
    "trans_month",
    "trans_dayofweek",
    "age"
]

X_train_numeric = X_train_smote[numeric_columns]
X_test_numeric = X_test[numeric_columns]

X_train_final = hstack([X_train_numeric, X_train_encoded])

X_test_final = hstack([X_test_numeric, X_test_encoded])

In [12]:
from collections import Counter

print("Przed SMOTENC:", Counter(y_train))
print("Po SMOTENC:", Counter(y_train_smote))

print("Train przed:", len(y_train))
print("Train po:", len(y_train_smote))

print("Test:", len(y_test))

Przed SMOTENC: Counter({0: 202695, 1: 1069})
Po SMOTENC: Counter({0: 202695, 1: 202695})
Train przed: 203764
Train po: 405390
Test: 135843


In [13]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    random_state=30,
    n_jobs=-1
)
model.fit(X_train_final, y_train_smote)

RandomForestClassifier(n_jobs=-1, random_state=30)

In [14]:
y_pred = model.predict(X_test_final)
y_pred_proba = model.predict_proba(X_test_final)[:, 1]

In [15]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    135130
           1       0.87      0.64      0.74       713

    accuracy                           1.00    135843
   macro avg       0.93      0.82      0.87    135843
weighted avg       1.00      1.00      1.00    135843



In [16]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Precision - jaka część transakcji oznaczonych przez model jako fraud serio nim byla
#
# Recall - jaka część wszystkich rzeczywistych fraudów
#          została wykryta przez model.
#
# F1-score - średnia harmoniczna precision i recall
#
# ROC-AUC - miara zdolności modelu do rozróżniania klas
#
# PR-AUC - pole pod krzywą Precision-Recall
#
# Confusion Matrix:
# TN - poprawnie rozpoznane transakcje normalne
# FP - normalne transakcje błędnie oznaczone jako fraud
# FN - fraudy, których model nie wykrył
# TP - poprawnie wykryte fraudy

report = classification_report(
    y_test,
    y_pred,
    target_names=["Normalna transakcja", "Fraud"],
    digits=4
)

print("                 WYNIKI MODELU RANDOMFOREST")

print("\n--- Classification Report ---")
print(report)

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("--- ROC-AUC ---")
print(f"{roc_auc:.4f}")


# PR-AUC
pr_auc = average_precision_score(y_test, y_pred_proba)

print("\n--- PR-AUC ---")
print(f"{pr_auc:.4f}")


# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("\n--- Confusion Matrix ---")
print(cm)

print("\nInterpretacja:")
print(f"  TN = {tn:>6}  | poprawnie rozpoznane normalne")
print(f"  FP = {fp:>6}  | normalne oznaczone jako fraud")
print(f"  FN = {fn:>6}  | niewykryte fraudy")
print(f"  TP = {tp:>6}  | poprawnie wykryte fraudy")

                 WYNIKI MODELU XGBOOST

--- Classification Report ---
                     precision    recall  f1-score   support

Normalna transakcja     0.9981    0.9995    0.9988    135130
              Fraud     0.8710    0.6438    0.7403       713

           accuracy                         0.9976    135843
          macro avg     0.9345    0.8216    0.8696    135843
       weighted avg     0.9975    0.9976    0.9975    135843

--- ROC-AUC ---
0.9818

--- PR-AUC ---
0.7827

--- Confusion Matrix ---
[[135062     68]
 [   254    459]]

Interpretacja:
  TN = 135062  | poprawnie rozpoznane normalne
  FP =     68  | normalne oznaczone jako fraud
  FN =    254  | niewykryte fraudy
  TP =    459  | poprawnie wykryte fraudy


Random Forest z wykorzystaniem SMOTENC osiągnął wyższy Recall (64,38%) niż model bazowy oraz wariant z RandomOverSamplerem, wykrywając większą liczbę rzeczywistych oszustw. Jednocześnie spadła precyzja (87,10%), a liczba fałszywych alarmów wzrosła do 68, co pokazuje kompromis między wykrywalnością oszustw a ograniczeniem błędnych klasyfikacji.

SMOTENC lepiej sprawdza się, jeśli priorytetem jest wykrycie jak największej liczby oszustw, natomiast RandomOverSampler zapewnia znacznie mniej fałszywych alarmów.